<a href="https://colab.research.google.com/github/ClaudioSeveroo/ciencia_de_dados/blob/main/exeraula8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!wget -q -O - https://dl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" > /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install -y google-chrome-stable

!pip install -U selenium

OK
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://dl.google.com/linux/chrome-stable/deb stable InRelease
Get:4 http://dl.google.com/linux/chrome/deb stable InRelease [2,548 B]
Hit:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:7 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Get:10 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,413 B]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Fetched 7,878 B in 1s (7,950 B/s)
Reading package lists... Done
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details

In [23]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

chrome_options = Options()
chrome_options.add_argument('--headless=new')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')

# Sem passar Service(...) — o Selenium Manager (4.6+) detecta o Chrome
# instalado e baixa o chromedriver compatível automaticamente.
driver = webdriver.Chrome(options=chrome_options)

In [24]:
import json

def buscar_temperatura(cidade):
    url = f"https://wttr.in/{cidade}?format=j1&lang=pt"
    driver.get(url)

    wait = WebDriverWait(driver, 15)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    texto = driver.find_element(By.TAG_NAME, "body").text

    try:
        dados = json.loads(texto)
    except json.JSONDecodeError:
        print("A resposta não veio em JSON. Conteúdo recebido:")
        print(texto[:500])
        return None

    hoje = dados["weather"][0]
    maxima = hoje["maxtempC"]
    minima = hoje["mintempC"]
    condicao = hoje["hourly"][4]["lang_pt"][0]["value"]  # condição por volta do meio-dia

    return {
        "cidade": cidade,
        "descricao": condicao,
        "maxima": f"{maxima}°C",
        "minima": f"{minima}°C"
    }

In [25]:
cidade = input("Digite o nome da cidade: ")
resultado = buscar_temperatura(cidade)

if resultado:
    print(f"\nCidade: {resultado['cidade']}")
    print(f"Condição: {resultado['descricao']}")
    print(f"Temperatura máxima: {resultado['maxima']}")
    print(f"Temperatura mínima: {resultado['minima']}")

driver.quit()

Digite o nome da cidade: Rio de Janeiro

Cidade: Rio de Janeiro
Condição: Ensolarado
Temperatura máxima: 25°C
Temperatura mínima: 18°C
